# Big Data Stack – Analiza, optymalizacja i bezpieczeństwo danych

Notatnik stanowi kontynuację skryptu `spark-hadoop.ipynb`. Zakładamy, że oczyszczone i spłaszczone dane pogodowe zostały już zapisane na HDFS w katalogu `/output/` w formatach Parquet (`data.parquet`, `data_v2.parquet`) oraz ORC (`data_v2.orc`).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from utils.config import load_config

In [ ]:
config = load_config("config.yaml")
sh_conf = config["spark_hadoop"]

spark_master = sh_conf["spark_master"]
hdfs_namenode = sh_conf["hdfs_namenode"]

hdfs_input = f"{hdfs_namenode}/input/weather.parquet"

hdfs_output_v1 = f"{hdfs_namenode}/output/data.parquet"
hdfs_output_v2 = f"{hdfs_namenode}/output/data_v2.parquet"
hdfs_output_orc = f"{hdfs_namenode}/output/data_v2.orc"
hdfs_output_part = f"{hdfs_namenode}/output/data_partitioned.parquet"
hdfs_output_anon = f"{hdfs_namenode}/output/data_anonymized.parquet"

In [ ]:
# Inicjalizacja sesji Spark
spark = (
    SparkSession.builder.appName("Spark Analiza i Cwiczenia")
    .master(spark_master)
    .getOrCreate()
)
spark

## Spark SQL i eksploracyjna analiza danych (EDA)

Wczytujemy przeliczone dane z pliku Parquet i rejestrujemy je jako widok tymczasowy, aby odpytywać je klasycznym SQL-em.

In [ ]:
df = spark.read.parquet(hdfs_input)
df.createOrReplaceTempView("pogoda")
df.printSchema()

In [ ]:
# 1. Rozkład zjawisk pogodowych i podstawowe statystyki temperatury oraz wiatru
spark.sql("""
    SELECT 
        weather_description,
        COUNT(*) AS liczba_pomiarow,
        ROUND(AVG(temp), 2) AS sr_temp,
        ROUND(MIN(temp_min), 2) AS min_temp,
        ROUND(MAX(temp_max), 2) AS max_temp,
        ROUND(AVG(wind_speed), 2) AS sr_wiatr
    FROM pogoda
    GROUP BY weather_description
    ORDER BY liczba_pomiarow DESC
""").show(truncate=False)

In [ ]:
# 2. Sprawdzenie braków danych (NULL) w poszczególnych kolumnach
spark.sql("""
    SELECT 
        COUNT(*) AS wszystkie_wiersze,
        COUNT(rain) AS wiersze_z_deszczem,
        COUNT(*) - COUNT(rain) AS braki_rain_1h,
        COUNT(*) - COUNT(wind_gust) AS braki_wind_gust
    FROM pogoda
""").show()

In [ ]:
# 3. Wykrywanie anomalii / wartości skrajnych (TOP 10 najsilniejszych porywów wiatru)
spark.sql("""
    SELECT 
        datetime,
        weather_description,
        temp,
        wind_speed,
        wind_gust
    FROM pogoda
    WHERE wind_gust IS NOT NULL
    ORDER BY wind_gust DESC
    LIMIT 10
""").show(truncate=False)

## DuckDB – lekka alternatywa OLAP bez klastra

Gdy zagregowane lub wyfiltrowane dane mieszczą się na jednej maszynie, nie potrzebujemy narzutu komunikacyjnego klastra Spark. Zapiszmy próbkę/wynik do lokalnego pliku Parquet i odpytajmy go bezpośrednio przez DuckDB.

In [ ]:
!pip install duckdb

In [ ]:
import duckdb

# Eksportujemy dane z Dataframe Sparka do lokalnego pliku Parquet na kontenerze Jupyter
local_parquet_path = "/tmp/pogoda_local.parquet"
df.toPandas().to_parquet(local_parquet_path, index=False)

In [ ]:
%%time

# Odpytanie pliku Parquet bezpośrednio przez silnik kolumnowy DuckDB (in-process)
duckdb.sql(f"""
    SELECT 
        weather_description,
        COUNT(*) AS n,
        ROUND(AVG(temp), 2) AS avg_temp,
        ROUND(AVG(humidity), 1) AS avg_humidity
    FROM '{local_parquet_path}'
    GROUP BY weather_description
    ORDER BY n DESC
""").df()

## Wizualizacja w Pythonie

**Zasada architektoniczna:** Nigdy nie robimy `.toPandas()` na surowych miliardach wierszy. Najpierw filtrujemy lub agregujemy dane na klastrze Sparka, a do pamięci lokalnej (drivera) ściągamy wyłącznie zagregowany lub próbkowany wynik pod wykres.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
# Krok 1: Agregacja w Sparku (słupki dla kategorii)
df_plot_cat = (
    df.groupBy("weather_description")
    .agg(F.count("*").alias("liczba"), F.avg("temp").alias("sr_temp"))
    .orderBy(F.col("liczba").desc())
    .toPandas()
)

plt.figure(figsize=(8, 4))
sns.barplot(data=df_plot_cat, x="liczba", y="weather_description", palette="Blues_r")
plt.title("Liczba obserwacji według kategorii pogody")
plt.xlabel("Liczba pomiarów")
plt.ylabel("Zjawisko (weather_main)")
plt.tight_layout()
plt.show()

In [ ]:
# Krok 2: Wykres pudełkowy (boxplot) dla rozkładów temperatur wg kategorii pogody
# Pobieramy tylko 2 potrzebne kolumny (column pruning w Parquet!)
df_plot_dist = df.select(
    "weather_description", "temp", "humidity", "pressure", "wind_speed"
).toPandas()

plt.figure(figsize=(9, 5))
sns.boxplot(data=df_plot_dist, x="temp", y="weather_description", palette="vlag")
plt.title("Rozkład temperatury (main_temp) dla poszczególnych zjawisk pogodowych")
plt.xlabel("Temperatura")
plt.ylabel("Zjawisko")
plt.tight_layout()
plt.show()

In [ ]:
# Krok 3: Heatmapa dla korelacji wybranych parametrów meteorologicznych
corr_matrix = df_plot_dist[
    ["temp", "humidity", "pressure", "wind_speed"]
].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Macierz korelacji parametrów pogodowych")
plt.tight_layout()
plt.show()

## Skalowalność, partycjonowanie i optymalizacja w Sparku

Sprawdzamy fizyczny podział na partycje, plany wykonania (`explain()`) z filtrowaniem u źródła (Predicate Pushdown), pamięć podręczną (`cache()`) oraz partycjonowanie katalogów na HDFS (`partitionBy`).

In [ ]:
# 1. Liczba partycji RDD dla wczytanych zbiorów (zwykły zapis vs coalesce(1))
df_v1 = spark.read.parquet(hdfs_output_v1)
df_v2 = spark.read.parquet(hdfs_output_v2)

print(f"Liczba partycji w data.parquet (v1):    {df_v1.rdd.getNumPartitions()}")
print(f"Liczba partycji w data_v2.parquet (v2): {df_v2.rdd.getNumPartitions()}")

In [ ]:
# 2. Analiza planu wykonania (EXPLAIN) - obserwujemy PushedFilters oraz ReadSchema
zapytanie_optymalne = df_v1.select("name", "main_temp", "weather_main").filter(
    (F.col("main_temp") > 20) & (F.col("weather_main") == "Clear")
)

zapytanie_optymalne.explain(extended=False)

In [ ]:
# 3. Partycjonowanie danych na dysku HDFS (Partitioning wg kolumny weather_main)
df_v1.write.mode("overwrite").partitionBy("weather_main").parquet(hdfs_output_part)

# Odczyt z wykorzystaniem Partition Pruning (Spark czyta tylko podkatalog weather_main=Rain)
df_rain = spark.read.parquet(hdfs_output_part).filter(F.col("weather_main") == "Rain")
df_rain.explain(extended=False)

In [ ]:
%%time
# 4. Caching w pamięci RAM (przydatne, gdy wielokrotnie odpytujemy ten sam przefiltrowany zbiór)
df_cached = df_v1.cache()
print("Pierwsze wywołanie (wczytanie z HDFS do RAM):", df_cached.count())

In [ ]:
%%time
print("Drugie wywołanie (bezpośrednio z pamięci RAM):", df_cached.count())

# Zwolnienie pamięci po zakończeniu analizy
df_cached.unpersist()

## Bezpieczeństwo danych – pseudonimizacja i anonimizacja

Zanim udostępnimy zbiór zespołom zewnętrznym lub wrzucimy go do środowiska testowego:
1. **Pseudonimizujemy** identyfikatory i nazwy stacji/lokalizacji za pomocą funkcji skrótu `SHA-256`.
2. **Anonimizujemy (generalizujemy)** dokładne współrzędne geograficzne GPS (`coord_lat`, `coord_lon`), zaokrąglając je do 1 miejsca po przecinku (dokładność ok. 11 km zamiast pojedynczej posesji).

In [ ]:
# 1. Pseudonimizacja i anonimizacja w PySpark
SALT = "tajny_klucz_soli_2026"

df_anon = (
    df_v1
    # Pseudonimizacja: zamiana nazwy miejscowości i ID na zahashowany identyfikator
    .withColumn("location_pseudo_id", F.sha2(F.concat(F.col("name"), F.lit(SALT)), 256))
    # Anonimizacja/generalizacja: obniżenie precyzji współrzędnych GPS
    .withColumn("coord_lat_approx", F.round(F.col("coord_lat"), 1))
    .withColumn("coord_lon_approx", F.round(F.col("coord_lon"), 1))
    # Usunięcie danych jawnych (bezpośrednich identyfikatorów)
    .drop("id", "name", "coord_lat", "coord_lon")
)

df_anon.select(
    "location_pseudo_id",
    "coord_lat_approx",
    "coord_lon_approx",
    "dt_ts",
    "main_temp",
    "weather_main",
).show(5, truncate=False)

In [ ]:
# Zapis zanonimizowanego zbioru na HDFS
df_anon.coalesce(1).write.mode("overwrite").parquet(hdfs_output_anon)

## Zadanie warsztatowe: Samodzielny mini-pipeline analityczny

**Cel zadania:**
1. Wczytaj dane z `/output/data.parquet`.
2. Usuń ewentualne duplikaty pomiarów dla tej samej lokalizacji (`name`) i czasu (`dt_ts`).
3. Dodaj nową kolumnę `roznica_temp_odczuwalnej` liczoną jako wartość bezwzględna różnicy między `main_temp` a `main_feels_like`.
4. Wyfiltruj tylko te obserwacje, gdzie wilgotność (`main_humidity`) przekracza 80%.
5. Znajdź 5 lokalizacji z największą średnią różnicą temperatury odczuwalnej i wyświetl wynik.

In [ ]:
# Miejsce na rozwiązanie uczestnika (poniżej gotowe rozwiązanie dla trenera):

df_wynik = (
    spark.read.parquet(hdfs_output_v1)
    .dropDuplicates(["name", "dt_ts"])
    .withColumn(
        "roznica_temp_odczuwalnej", F.abs(F.col("main_temp") - F.col("main_feels_like"))
    )
    .filter(F.col("main_humidity") > 80)
    .groupBy("name")
    .agg(
        F.count("*").alias("liczba_pomiarow"),
        F.round(F.avg("roznica_temp_odczuwalnej"), 2).alias("sr_roznica_odczuwalna"),
        F.round(F.avg("wind_speed"), 2).alias("sr_wiatr"),
    )
    .orderBy(F.col("sr_roznica_odczuwalna").desc())
    .limit(5)
)

df_wynik.show(truncate=False)

In [ ]:
spark.stop()